# 07 - Train Per-Crop Disease Classifier (EfficientNet-B2)

Stage 7: Train one disease classifier per crop (EfficientNet-B2).

Reads disease_dataset/<crop>/{train,val,test}/<disease>/ produced by
06_prepare_disease_dataset.py. Trains ONE crop at a time (pass --crop),
or all five sequentially if you leave --crop out -- useful since your
GPU will be busy a while and you may want to run crops one at a time
between other work.

Saves each crop's model + label list separately:
    models/disease_<crop>.pth
    models/disease_<crop>_labels.json

Set CROP_TO_TRAIN below and run -- set it to None to train all 5 crops
back-to-back in one run (only do this if your GPU has time to spare;
each crop takes as long as 02_train_crop_identifier.py did per epoch).

Install deps (same as 02_train_crop_identifier.py):
    pip install torch torchvision timm albumentations scikit-learn --break-system-packages

## Imports & Configuration

In [1]:
import json
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, WeightedRandomSampler, Dataset
from torchvision.datasets import ImageFolder
import albumentations as A
from albumentations.pytorch import ToTensorV2
import timm
from sklearn.metrics import classification_report
import cv2

DATA_ROOT = Path("disease_dataset")
MODELS_DIR = Path("models")
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 25
LR = 3e-4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CROPS = ["Cotton", "Groundnut", "Pepper Bell", "Potato", "Tomato"]

train_tf = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.2),
    A.RandomRotate90(p=0.5),
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05, p=0.5),
    A.GaussianBlur(blur_limit=(3, 5), p=0.2),
    A.CoarseDropout(num_holes_range=(1, 4), hole_height_range=(12, 24), hole_width_range=(12, 24), p=0.3),  # simulates leaf-spot occlusion
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

eval_tf = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

z:\Crop Identification\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## `AlbumentationsImageFolder`

In [2]:
class AlbumentationsImageFolder(Dataset):
    def __init__(self, root, transform):
        self.base = ImageFolder(root)
        self.transform = transform
        self.classes = self.base.classes

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        path, label = self.base.samples[idx]
        image = cv2.imread(path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = self.transform(image=image)["image"]
        return image, label

## `build_weighted_sampler`

In [3]:
def build_weighted_sampler(dataset):
    targets = [label for _, label in dataset.base.samples]
    class_counts = np.bincount(targets)
    class_weights = 1.0 / class_counts
    sample_weights = [class_weights[t] for t in targets]
    return WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

## `train_one_crop`

In [4]:
from tqdm.auto import tqdm

def train_one_crop(crop_name):
    crop_slug = crop_name.replace(" ", "_")
    train_dir = DATA_ROOT / crop_slug / "train"
    val_dir = DATA_ROOT / crop_slug / "val"

    if not train_dir.exists():
        print(f"Skipping {crop_name}: {train_dir} not found (did 06 run for this crop?)")
        return

    print(f"\n{'=' * 20} Training disease classifier: {crop_name} {'=' * 20}")

    train_ds = AlbumentationsImageFolder(train_dir, train_tf)
    val_ds = AlbumentationsImageFolder(val_dir, eval_tf)
    num_classes = len(train_ds.classes)
    print(f"Classes ({num_classes}): {train_ds.classes}")

    if num_classes < 2:
        print(f"Skipping {crop_name}: fewer than 2 surviving classes after pruning in step 06.")
        return

    sampler = build_weighted_sampler(train_ds)
    
    # Set num_workers=0 to prevent Windows hanging/freezing issues
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    model = timm.create_model("efficientnet_b2", pretrained=True, num_classes=num_classes)
    model.to(DEVICE)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    criterion = nn.CrossEntropyLoss()

    best_val_acc = 0.0
    patience, patience_counter = 5, 0
    model_path = MODELS_DIR / f"disease_{crop_slug}.pth"
    labels_path = MODELS_DIR / f"disease_{crop_slug}_labels.json"
    MODELS_DIR.mkdir(parents=True, exist_ok=True)

    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0.0
        
        # 1. Training Loop with Live Progress Bar
        train_pbar = tqdm(train_loader, desc=f"  [{crop_name}] Epoch {epoch+1}/{EPOCHS} Train", leave=False)
        for images, labels in train_pbar:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * images.size(0)
            train_pbar.set_postfix({"loss": f"{loss.item():.4f}"})
            
        scheduler.step()
        train_loss = running_loss / len(train_ds)

        # 2. Validation Loop with Live Progress Bar
        model.eval()
        correct, total = 0, 0
        all_preds, all_labels = [], []
        
        val_pbar = tqdm(val_loader, desc=f"  [{crop_name}] Epoch {epoch+1}/{EPOCHS} Val  ", leave=False)
        with torch.no_grad():
            for images, labels in val_pbar:
                images, labels = images.to(DEVICE), labels.to(DEVICE)
                outputs = model(images)
                preds = outputs.argmax(dim=1)
                correct += (preds == labels).sum().item()
                total += labels.size(0)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
        val_acc = correct / total

        print(f"[{crop_name}] Epoch {epoch+1}/{EPOCHS} - train_loss: {train_loss:.4f} - val_acc: {val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            patience_counter = 0
            torch.save(model.state_dict(), model_path)
            with open(labels_path, "w") as f:
                json.dump(train_ds.classes, f)
            print(f"  -> saved new best model (val_acc={val_acc:.4f})")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"[{crop_name}] Early stopping triggered.")
                break

    print(f"[{crop_name}] Best val_acc: {best_val_acc:.4f}. Model saved to {model_path}")

    print(f"\n[{crop_name}] Val-set classification report (last epoch):")
    print(classification_report(all_labels, all_preds, target_names=train_ds.classes, digits=3, zero_division=0))

## Run

In [5]:
CROP_TO_TRAIN = "Tomato"  # set to one of CROPS, or None to train all 5

crops_to_run = [CROP_TO_TRAIN] if CROP_TO_TRAIN else CROPS
for crop in crops_to_run:
    train_one_crop(crop)


==================== Training disease classifier: Tomato ====================
Classes (8): ['BS', 'EB', 'H', 'LB', 'ML', 'MV', 'S', 'YCV']


[Tomato] Epoch 1/25 - train_loss: 0.8561 - val_acc: 0.8384
  -> saved new best model (val_acc=0.8384)


[Tomato] Epoch 2/25 - train_loss: 0.3099 - val_acc: 0.8750
  -> saved new best model (val_acc=0.8750)


[Tomato] Epoch 3/25 - train_loss: 0.2261 - val_acc: 0.8869
  -> saved new best model (val_acc=0.8869)


[Tomato] Epoch 4/25 - train_loss: 0.1494 - val_acc: 0.8825


...

[Tomato] Epoch 21/25 - train_loss: 0.0079 - val_acc: 0.9267


[Tomato] Epoch 22/25 - train_loss: 0.0066 - val_acc: 0.9246


[Tomato] Epoch 23/25 - train_loss: 0.0111 - val_acc: 0.9235


[Tomato] Epoch 24/25 - train_loss: 0.0134 - val_acc: 0.9289
  -> saved new best model (val_acc=0.9289)


[Tomato] Epoch 25/25 - train_loss: 0.0099 - val_acc: 0.9300
  -> saved new best model (val_acc=0.9300)
[Tomato] Best val_acc: 0.9300. Model saved to models\disease_Tomato.pth

[Tomato] Val-set classification report (last epoch):
              precision    recall  f1-score   support

          BS      0.988     0.934     0.961       183
          EB      0.940     0.932     0.936       219
           H      0.953     0.980     0.966       205
          LB      0.914     0.958     0.935       166
          ML      0.920     0.920     0.920        75
          MV      0.750     0.429     0.545        14
           S      0.761     0.778     0.769        45
         YCV      0.750     0.857     0.800        21

    accuracy                          0.930       928
   macro avg      0.872     0.848     0.854       928
weighted avg      0.930     0.930     0.929       928



In [6]:
CROP_TO_TRAIN = "Cotton"  # set to one of CROPS, or None to train all 5

crops_to_run = [CROP_TO_TRAIN] if CROP_TO_TRAIN else CROPS
for crop in crops_to_run:
    train_one_crop(crop)


==================== Training disease classifier: Cotton ====================
Classes (8): ['ALS', 'BB', 'CV', 'FW', 'H', 'PM', 'TS', 'VW']


[Cotton] Epoch 1/25 - train_loss: 0.8958 - val_acc: 0.8051
  -> saved new best model (val_acc=0.8051)


[Cotton] Epoch 2/25 - train_loss: 0.3118 - val_acc: 0.8484
  -> saved new best model (val_acc=0.8484)


[Cotton] Epoch 3/25 - train_loss: 0.2603 - val_acc: 0.8917
  -> saved new best model (val_acc=0.8917)


[Cotton] Epoch 4/25 - train_loss: 0.0947 - val_acc: 0.9206
  -> saved new best model (val_acc=0.9206)


...

[Cotton] Epoch 8/25 - train_loss: 0.0690 - val_acc: 0.9567


[Cotton] Epoch 9/25 - train_loss: 0.0429 - val_acc: 0.9350


[Cotton] Epoch 10/25 - train_loss: 0.0586 - val_acc: 0.9386


[Cotton] Epoch 11/25 - train_loss: 0.0455 - val_acc: 0.9422


[Cotton] Epoch 12/25 - train_loss: 0.0273 - val_acc: 0.9495
[Cotton] Early stopping triggered.
[Cotton] Best val_acc: 0.9603. Model saved to models\disease_Cotton.pth

[Cotton] Val-set classification report (last epoch):
              precision    recall  f1-score   support

         ALS      0.962     0.962     0.962        26
          BB      0.950     0.884     0.916        43
          CV      0.857     1.000     0.923        12
          FW      0.966     0.966     0.966        89
           H      1.000     0.946     0.972        56
          PM      0.833     1.000     0.909         5
          TS      0.800     0.667     0.727         6
          VW      0.909     1.000     0.952        40

    accuracy                          0.949       277
   macro avg      0.910     0.928     0.916       277
weighted avg      0.951     0.949     0.949       277



In [7]:
CROP_TO_TRAIN = "Groundnut"  # set to one of CROPS, or None to train all 5

crops_to_run = [CROP_TO_TRAIN] if CROP_TO_TRAIN else CROPS
for crop in crops_to_run:
    train_one_crop(crop)


==================== Training disease classifier: Groundnut ====================
Classes (6): ['ALS', 'H', 'LS', 'ND', 'R', 'Ros']


[Groundnut] Epoch 1/25 - train_loss: 0.4552 - val_acc: 0.8256
  -> saved new best model (val_acc=0.8256)


[Groundnut] Epoch 2/25 - train_loss: 0.1779 - val_acc: 0.8285
  -> saved new best model (val_acc=0.8285)


[Groundnut] Epoch 3/25 - train_loss: 0.1182 - val_acc: 0.8934
  -> saved new best model (val_acc=0.8934)


[Groundnut] Epoch 4/25 - train_loss: 0.1253 - val_acc: 0.8948
  -> saved new best model (val_acc=0.8948)


...

[Groundnut] Epoch 12/25 - train_loss: 0.0448 - val_acc: 0.9179


[Groundnut] Epoch 13/25 - train_loss: 0.0375 - val_acc: 0.9164


[Groundnut] Epoch 14/25 - train_loss: 0.0236 - val_acc: 0.9179


[Groundnut] Epoch 15/25 - train_loss: 0.0203 - val_acc: 0.9179


[Groundnut] Epoch 16/25 - train_loss: 0.0180 - val_acc: 0.9193
[Groundnut] Early stopping triggered.
[Groundnut] Best val_acc: 0.9236. Model saved to models\disease_Groundnut.pth

[Groundnut] Val-set classification report (last epoch):
              precision    recall  f1-score   support

         ALS      0.982     0.982     0.982        57
           H      0.891     0.938     0.914       227
          LS      0.936     0.895     0.915       294
          ND      0.957     0.880     0.917        50
           R      0.875     0.942     0.907        52
         Ros      0.867     0.929     0.897        14

    accuracy                          0.919       694
   macro avg      0.918     0.928     0.922       694
weighted avg      0.921     0.919     0.919       694



In [8]:
CROP_TO_TRAIN = "Pepper Bell"  # set to one of CROPS, or None to train all 5

crops_to_run = [CROP_TO_TRAIN] if CROP_TO_TRAIN else CROPS
for crop in crops_to_run:
    train_one_crop(crop)


==================== Training disease classifier: Pepper Bell ====================
Classes (7): ['BS', 'CLS', 'E', 'H', 'LC', 'ND', 'PM']


[Pepper Bell] Epoch 1/25 - train_loss: 0.2279 - val_acc: 0.9901
  -> saved new best model (val_acc=0.9901)


[Pepper Bell] Epoch 2/25 - train_loss: 0.0349 - val_acc: 0.9910
  -> saved new best model (val_acc=0.9910)


[Pepper Bell] Epoch 3/25 - train_loss: 0.0501 - val_acc: 0.9847


[Pepper Bell] Epoch 4/25 - train_loss: 0.0350 - val_acc: 0.9955
  -> saved new best model (val_acc=0.9955)


[Pepper Bell] Epoch 5/25 - train_loss: 0.0148 - val_acc: 0.9901


[Pepper Bell] Epoch 6/25 - train_loss: 0.0164 - val_acc: 0.9955


[Pepper Bell] Epoch 7/25 - train_loss: 0.0158 - val_acc: 0.9928


[Pepper Bell] Epoch 8/25 - train_loss: 0.0211 - val_acc: 0.9946


[Pepper Bell] Epoch 9/25 - train_loss: 0.0143 - val_acc: 0.9892
[Pepper Bell] Early stopping triggered.
[Pepper Bell] Best val_acc: 0.9955. Model saved to models\disease_Pepper_Bell.pth

[Pepper Bell] Val-set classification report (last epoch):
              precision    recall  f1-score   support

          BS      1.000     0.987     0.993       535
         CLS      1.000     0.995     0.998       210
           E      1.000     0.833     0.909         6
           H      0.970     0.991     0.980       227
          LC      0.962     1.000     0.981        51
          ND      0.983     1.000     0.991        58
          PM      0.929     0.963     0.945        27

    accuracy                          0.989      1114
   macro avg      0.978     0.967     0.971      1114
weighted avg      0.990     0.989     0.989      1114



In [9]:
CROP_TO_TRAIN = "Potato"  # set to one of CROPS, or None to train all 5

crops_to_run = [CROP_TO_TRAIN] if CROP_TO_TRAIN else CROPS
for crop in crops_to_run:
    train_one_crop(crop)


==================== Training disease classifier: Potato ====================
Classes (8): ['B', 'EB', 'F', 'H', 'LB', 'N', 'P', 'V']


[Potato] Epoch 1/25 - train_loss: 0.7737 - val_acc: 0.8688
  -> saved new best model (val_acc=0.8688)


[Potato] Epoch 2/25 - train_loss: 0.2572 - val_acc: 0.9209
  -> saved new best model (val_acc=0.9209)


[Potato] Epoch 3/25 - train_loss: 0.1621 - val_acc: 0.9322
  -> saved new best model (val_acc=0.9322)


[Potato] Epoch 4/25 - train_loss: 0.1263 - val_acc: 0.9418
  -> saved new best model (val_acc=0.9418)


...

[Potato] Epoch 6/25 - train_loss: 0.0822 - val_acc: 0.9366


[Potato] Epoch 7/25 - train_loss: 0.0842 - val_acc: 0.9453


[Potato] Epoch 8/25 - train_loss: 0.0555 - val_acc: 0.9427


[Potato] Epoch 9/25 - train_loss: 0.0501 - val_acc: 0.9461


[Potato] Epoch 10/25 - train_loss: 0.0317 - val_acc: 0.9435
[Potato] Early stopping triggered.
[Potato] Best val_acc: 0.9461. Model saved to models\disease_Potato.pth

[Potato] Val-set classification report (last epoch):
              precision    recall  f1-score   support

           B      0.943     0.976     0.960        85
          EB      0.996     1.000     0.998       265
           F      0.848     0.817     0.832       109
           H      0.969     0.964     0.966       224
          LB      0.979     0.972     0.976       290
           N      0.900     0.818     0.857        11
           P      0.835     0.798     0.816        89
           V      0.826     0.910     0.866        78

    accuracy                          0.944      1151
   macro avg      0.912     0.907     0.909      1151
weighted avg      0.944     0.944     0.943      1151

